# 020. LSTM/GRU input/output shape

- return_sequences = False, True 일 때의 output 비교

- return_state = False, True 일 때의 internal state output 비교

- Bidirectional LSTM/GRU 의 output 비교

In [5]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, GRU
import numpy as np

T = 30   #Time Steps
D = 1   #features
U = 3   #LSTM units

X = np.random.randn(4, T, D)
print(X.shape)

(4, 30, 1)


# LSTM

## return_sequences

- False (default) - last time step 의 output 만 반환
- True - 모든 timestep 의 output 을 모두 반환

In [6]:
def lstm(return_sequences=False):
    input_ = Input(shape=(T, D))
    rnn = LSTM(U, return_sequences=return_sequences)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)

    return model.predict(X)

print("---- return_sequences=False ----> last timestep 의 output 만 반환")
lstm_out = lstm(return_sequences=False)
print(lstm_out.shape)
print(lstm_out)

print("\n---- return_sequences=True ----> 모든 timestep 별 output 출력")
lstm_out = lstm(return_sequences=True)
print(lstm_out.shape)
print(lstm_out)

---- return_sequences=False ----> last timestep 의 output 만 반환
(4, 3)
[[-0.02507054 -0.05451694  0.03048269]
 [ 0.1843272   0.03315487 -0.13205552]
 [ 0.05451262 -0.03746881 -0.0081749 ]
 [-0.3485176  -0.13520858  0.24972817]]

---- return_sequences=True ----> 모든 timestep 별 output 출력
(4, 30, 3)
[[[ 3.19353610e-01 -5.67853153e-02 -9.50454399e-02]
  [ 3.02265853e-01 -1.03816867e-01 -1.27302766e-01]
  [ 3.64233762e-01 -1.26094788e-01 -1.80477157e-01]
  [ 2.65161991e-01 -1.32092819e-01 -1.27694219e-01]
  [ 2.18489096e-01 -1.05427481e-01 -9.82060283e-02]
  [ 1.62240684e-01 -7.72231519e-02 -6.42003268e-02]
  [ 2.30225250e-01 -8.45259279e-02 -1.06321923e-01]
  [ 4.00112122e-02  2.13263221e-02  6.23268113e-02]
  [ 2.64748275e-01 -4.19965237e-02 -1.09843258e-02]
  [ 1.23933688e-01 -3.18611078e-02  1.13697303e-02]
  [ 2.95722187e-02  1.26428893e-02  5.14515713e-02]
  [ 5.16887121e-02 -4.43224283e-03  2.26530731e-02]
  [ 8.79459381e-02 -2.29263213e-02 -1.26542924e-02]
  [ 4.13663059e-01 -6.8404153

## return_state

- False (default) - output 만 반환

- True - output, last step 의 hidden state, cell state (LSTM 의 경우) 반환

In [15]:
def lstm(return_state=False):
    input_ = Input(shape=(T, D))
    rnn = LSTM(U, return_state=return_state)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:
        o, h, c = model.predict(X)
        print("o :", o.shape)
        print("h :", h.shape)
        print("c :", c.shape)
    else:
        o = model.predict(X)
        print("o :", o.shape)
        print(o)

print("---- return_state=False ----> outout only")       
lstm(return_state=False)
print("\n---- return_state=True ----> outout, hidden state, cell state all")      
lstm(return_state=True)

---- return_state=False ----> outout only
o : (4, 3)
[[-0.04140986 -0.12326118 -0.00311707]
 [-0.13287981 -0.15473787 -0.03382233]
 [-0.07616539 -0.19256942 -0.00594227]
 [ 0.09133799  0.02211378  0.01997466]]

---- return_state=True ----> outout, hidden state, cell state all
o : (4, 3)
h : (4, 3)
c : (4, 3)


# Bidirectional LSTM

- 순방향, 역방향이 concatenate 된 output 출력  

- hidden state, cell state 는 순방향, 역방향 별도 출력

In [9]:
T, D, U

(30, 1, 3)

In [11]:
def bi_lstm(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = Bidirectional(LSTM(U, return_state=return_state, return_sequences=return_sequences))
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:    
        o, h1, c1, h2, c2 = model.predict(X)
        print("o :",o.shape)
        print("h1 :", h1.shape)
        print("c1 :", c1.shape)
        print("h2 :", h2.shape)
        print("c2 :", c2.shape)
    else:
        o = model.predict(X)
        print("o :", o.shape)

print("---- 순방향, 역방향이 concatenate 된 many-to-one output")
bi_lstm(return_sequences=False, return_state=False)
print()
print("---- 순방향, 역방향이 concatenate 된 many-to-many output")
bi_lstm(return_sequences=True)
print()
print("---- 순방향, 역방향이 concatenate 되고 return_state=True인  many-to-one output")
bi_lstm(return_state=True)

---- 순방향, 역방향이 concatenate 된 many-to-one output
o : (4, 6)

---- 순방향, 역방향이 concatenate 된 many-to-many output
o : (4, 30, 6)

---- 순방향, 역방향이 concatenate 되고 return_state=True인  many-to-one output
o : (4, 6)
h1 : (4, 3)
c1 : (4, 3)
h2 : (4, 3)
c2 : (4, 3)


# GRU 

- cell state 가 없는 것만 LSTM 과 차이

In [13]:
def gru(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = GRU(U, return_state=return_state, return_sequences=return_sequences)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:    
        o, h = model.predict(X)
        print("o :", o.shape)
        print("h :", h.shape)
    else:
        o = model.predict(X)
        print("o :", o.shape)

print("---- Many-to-One output ----")
gru(return_sequences=False, return_state=False)
print()
print("---- Many-to-Many output ----")
gru(return_sequences=True)
print()
print("---- Sequence-to-Vector output ----")
gru(return_state=True)

---- Many-to-One output ----
o : (4, 3)

---- Many-to-Many output ----
o : (4, 30, 3)

---- Sequence-to-Vector output ----
o : (4, 3)
h : (4, 3)


# Bidirectional GRU

- cell state 가 없는 것 외에 LSTM 과 동일

In [14]:
def bi_gru(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = Bidirectional(GRU(U, return_state=return_state, return_sequences=return_sequences))
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    if return_state:    
        o, h1, h2 = model.predict(X)
        print("o :", o.shape)
        print("h1 :", h1.shape)
        print("h2 :", h2.shape)
    else:
        o = model.predict(X)
        print("o :", o.shape)
        
print("---- 순방향, 역방향이 concatenate 된 many-to-one output")
bi_gru(return_sequences=False, return_state=False)
print()
print("---- 순방향, 역방향이 concatenate 된 many-to-many output")
bi_gru(return_sequences=True)
print()
print("---- 순방향, 역방향이 concatenate 된 sequence-to-vector output")
bi_gru(return_state=True)

---- 순방향, 역방향이 concatenate 된 many-to-one output
o : (4, 6)

---- 순방향, 역방향이 concatenate 된 many-to-many output
o : (4, 30, 6)

---- 순방향, 역방향이 concatenate 된 sequence-to-vector output
o : (4, 6)
h1 : (4, 3)
h2 : (4, 3)
